![cyber_photo](cyber_photo.jpg)

Cyber threats are a growing concern for organizations worldwide. These threats take many forms, including malware, phishing, and denial-of-service (DOS) attacks, compromising sensitive information and disrupting operations. The increasing sophistication and frequency of these attacks make it imperative for organizations to adopt advanced security measures. Traditional threat detection methods often fall short due to their inability to adapt to new and evolving threats. This is where deep learning models come into play.

Deep learning models can analyze vast amounts of data and identify patterns that may not be immediately obvious to human analysts. By leveraging these models, organizations can proactively detect and mitigate cyber threats, safeguarding their sensitive information and ensuring operational continuity.

As a cybersecurity analyst, you identify and mitigate these threats. In this project, you will design and implement a deep learning model to detect cyber threats. The BETH dataset simulates real-world logs, providing a rich source of information for training and testing your model. The data has already undergone preprocessing, and we have a target label, `sus_label`, indicating whether an event is malicious (1) or benign (0).

By successfully developing this model, you will contribute to enhancing cybersecurity measures and protecting organizations from potentially devastating cyber attacks.


### The Data

| Column     | Description              |
|------------|--------------------------|
|`processId`|The unique identifier for the process that generated the event - int64 |
|`threadId`|ID for the thread spawning the log - int64|
|`parentProcessId`|Label for the process spawning this log - int64|
|`userId`|ID of user spawning the log|Numerical - int64|
|`mountNamespace`|Mounting restrictions the process log works within - int64|
|`argsNum`|Number of arguments passed to the event - int64|
|`returnValue`|Value returned from the event log (usually 0) - int64|
|`sus_label`|Binary label as suspicous event (1 is suspicious, 0 is not) - int64|

More information on the dataset: [BETH dataset](accreditation.md)

In [113]:
# Import required libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torchmetrics import Accuracy
# from sklearn.metrics import accuracy_score  # uncomment to use sklearn

In [114]:
# Load preprocessed data
train_df = pd.read_csv('labelled_train.csv')
test_df = pd.read_csv('labelled_test.csv')
val_df = pd.read_csv('labelled_validation.csv')

# View the first 5 rows of training set
train_df.head()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,7337,1,100,4026532231,5,0,1
1,381,7337,1,100,4026532231,1,0,1
2,381,7337,1,100,4026532231,0,0,1
3,7347,7347,7341,0,4026531840,2,-2,1
4,7347,7347,7341,0,4026531840,4,0,1


In [115]:
# In Google Cbloud Datalab, you can add a new cell by using keyboard shortcuts:
# - Press 'A' to add a cell above the current cell.
# - Press 'B' to add a cell below the current cell.
# Alternatively, use the menu: Edit > Insert Cell Above/Below.

# This cell is just a reminder; no code is needed to add a new cell.

In [116]:
val_df.head()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,381,1,101,4026532232,3,15,0
1,378,378,1,100,4026532231,3,15,0
2,1,1,0,0,4026531840,4,0,0
3,1,1,0,0,4026531840,4,12,0
4,1,1,0,0,4026531840,2,0,0


In [117]:
# Check for missing values in each dataset
print("Missing values in Training set:\n", train_df.isnull().sum())
print("\nMissing values in Validation set:\n", val_df.isnull().sum())
print("\nMissing values in Test set:\n", test_df.isnull().sum())

Missing values in Training set:
 processId          0
threadId           0
parentProcessId    0
userId             0
mountNamespace     0
argsNum            0
returnValue        0
sus_label          0
dtype: int64

Missing values in Validation set:
 processId          0
threadId           0
parentProcessId    0
userId             0
mountNamespace     0
argsNum            0
returnValue        0
sus_label          0
dtype: int64

Missing values in Test set:
 processId          0
threadId           0
parentProcessId    0
userId             0
mountNamespace     0
argsNum            0
returnValue        0
sus_label          0
dtype: int64


In [118]:
#define train and test datasets
X_train = train_df.drop('sus_label',axis= 1).values
X_test = test_df.drop('sus_label',axis= 1).values
X_val = val_df.drop('sus_label',axis= 1).values
y_train = train_df['sus_label'].values
y_test = test_df['sus_label'].values
y_val = val_df['sus_label'].values






In [119]:
scaler = StandardScaler()

In [120]:
#fit the scaler on the training data
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

X_val = scaler.fit_transform(X_val)

In [121]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

In [122]:
## Create DataLoader for training and validation
train_data = TensorDataset(X_train_tensor, y_train_tensor)
val_data = TensorDataset(X_val_tensor, y_val_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_data, shuffle=True, batch_size=64)
val_loader = DataLoader(val_data, shuffle=False, batch_size=64)

In [123]:
print(X_train_tensor.shape)

torch.Size([763144, 7])


In [124]:
print(X_train.shape[1])

7


In [125]:
model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1],128),
    nn.ReLU(),
    nn.Linear(128,64),
    nn.ReLU(),
    nn.Linear(64,1),
    nn.Sigmoid()
              
)

In [126]:
criterion = nn.BCELoss()

In [127]:
optimizer = optim.Adam(model.parameters(),lr = 0.001, weight_decay = 0.0001)

In [128]:
print(y_train_tensor.shape)

torch.Size([763144])


In [129]:
accuracy = Accuracy(task = 'binary', threshold = 0.5)

In [130]:
num_ep = 10
best_val_accuracy = 0.0

for epoch in range(num_ep):
    model.train()
    running_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        outputs = model(X_batch).squeeze() 
        
        loss = criterion(outputs, y_batch)
        
        loss.backward()
        
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    model.eval()
    current_val_acc = 0.0  
    
    with torch.no_grad():
        
        for X_val_batch, y_val_batch in val_loader:
            
            val_outputs = model(X_val_batch).squeeze()
            
            val_predictions = (val_outputs >= 0.5).float()
            
            
            current_val_acc += accuracy(val_predictions, y_val_batch).item()

    current_val_acc /= len(val_loader)

    if current_val_acc > best_val_accuracy:
        
        best_val_accuracy = current_val_acc


    print(f'Epoch {epoch+1}/{num_ep}, Loss: {epoch_loss:.4f}, Val Acc: {current_val_acc:.4f}')

# Final Output
final_score = int(best_val_accuracy * 100)
print(f'Best Validation Accuracy achieved: {final_score}%')

Epoch 1/10, Loss: 0.0040, Val Acc: 1.0000
Epoch 2/10, Loss: 0.0023, Val Acc: 1.0000
Epoch 3/10, Loss: 0.0023, Val Acc: 1.0000
Epoch 4/10, Loss: 0.0023, Val Acc: 1.0000
Epoch 5/10, Loss: 0.0024, Val Acc: 1.0000
Epoch 6/10, Loss: 0.0024, Val Acc: 0.9999
Epoch 7/10, Loss: 0.0024, Val Acc: 1.0000
Epoch 8/10, Loss: 0.0023, Val Acc: 1.0000
Epoch 9/10, Loss: 0.0023, Val Acc: 0.9998
Epoch 10/10, Loss: 0.0023, Val Acc: 0.9999
Best Validation Accuracy achieved: 99%
